In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 25 samples and 11825 columns.
First few rows of the data:


,os.time,os.event,pfs.time,pfs.event,samples,sample.timepoint,ajcc.stage,tumor.response,type.of.lesion,type.of.primary,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1,ZZZ3
0,92.9,0,48.1,1,MM909_01_1,Before TILs,M1c,-100.00,LN,unknown,...,3.14586,2.20174,2.70930,0.82517,1.36987,2.76248,2.55180,5.53388,2.56353,3.15532
1,6.9,1,2.0,1,MM909_02_1,Before TILs,M1b,10.94,LN,skin,...,3.36122,3.40559,2.97088,1.27695,1.29983,2.69095,2.45870,5.42459,1.80400,3.68111
2,11.4,1,3.8,1,MM909_03_1,Before TILs,M1c,11.92,SC,unknown,...,3.29989,2.79518,3.91229,1.19076,2.07418,2.84976,3.43125,5.08218,2.69648,4.74777
3,4.6,1,2.3,1,MM909_06_1,Before TILs,M1c,-7.14,LN,skin,...,3.00165,3.22395,4.48321,0.38875,0.91491,1.91299,2.96821,5.00434,0.54810,3.79871
4,71.5,0,13.1,1,MM909_11_1,Before TILs,M1a,-100.00,LN,skin,...,2.73279,3.54445,3.98488,0.71845,1.15190,2.64073,2.92381,5.07525,2.02028,3.90744


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for os.time or os.event: 0
Number of samples with complete data for both variables: 25
Number of censored samples (event_var == 0.0): 8
Number of events (event_var == 1.0): 17
Censored ratio (among complete): 8 / 25 = 0.320 (32.0%)
Number of samples with incomplete data for pfs.time or pfs.event: 0
Number of samples with complete data for both variables: 25
Number of censored samples (event_var == 0.0): 5
Number of events (event_var == 1.0): 20
Censored ratio (among complete): 5 / 25 = 0.200 (20.0%)
Updated stats for 2 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE100797/description.json
